[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/25_mha_solution.ipynb)

# 🟡 Solution: Multi-Head Attention (nnx.Module)

*Attention & Transformers · Medium*

Reference implementation. Try it yourself in `25_mha.ipynb` first.

---
Implement **multi-head self-attention** as a `flax.nnx.Module`.

$$\text{head}_h = \operatorname{softmax}\!\left(\frac{Q_h K_h^\top}{\sqrt{d_h}}\right)V_h,
\qquad
\text{out} = \big[\text{head}_1 \,\|\, \cdots \,\|\, \text{head}_H\big]\,W_o$$

with $Q = xW_q$, $K = xW_k$, $V = xW_v$ and $d_h = d_{model}/H$.

### Signature
- `MultiHeadAttention(d_model, num_heads, *, rngs: nnx.Rngs)`
- `__call__(x, mask=None)` maps `(B, T, d_model) -> (B, T, d_model)`
- `mask` is boolean, broadcastable to `(B, H, T, T)`; `True` = attend

### Rules
- Subclass `nnx.Module`; no `nnx.MultiHeadAttention`, no `jax.nn.dot_product_attention`
- Parameters named exactly `w_q`, `w_k`, `w_v`, `w_o`, each an `nnx.Param` of
  shape `(d_model, d_model)` — JAX layout is `(din, dout)` and you compute `x @ W`
- No biases on the projections
- Draw each matrix with `jax.random.normal(rngs.params(), ...) / sqrt(d_model)`
- Scale scores by $\sqrt{d_h}$, the **per-head** dim
- Raise or assert if `d_model % num_heads != 0`

### The reshape trap
The projected tensor is `(B, T, d_model)` and you want `(B, H, T, d_h)`. There is
exactly one correct route:

```python
x.reshape(B, T, H, d_head).transpose(0, 2, 1, 3)     # correct
x.reshape(B, H, T, d_head)                            # silently wrong
```

`reshape` reads memory in row-major order, so the last axis `d_model` splits
naturally into `(H, d_h)` — head $h$ owns the contiguous slice
`[h*d_h : (h+1)*d_h]` of the feature vector. Reshaping straight to `(B, H, T, d_h)`
instead slices along the **token** axis, so "head 0" ends up holding the first
`T/H` tokens' full feature vectors. The shapes typecheck, the loss goes down a
bit, and the model is quietly broken. Coming back the other way you must
`transpose` before `reshape` for the same reason.

### Why heads are free
Count parameters: $4d_{model}^2$, with no $H$ anywhere. Count FLOPs for the score
matrix: $B \cdot H \cdot T^2 \cdot d_h = B \cdot T^2 \cdot d_{model}$ — again no
$H$. Splitting into heads is a *reinterpretation* of the same matrix, not extra
work. What you buy is $H$ independent similarity subspaces: one head can track
syntactic agreement while another tracks positional offset, instead of a single
softmax being forced to average those signals into one distribution. The cost is
that each head sees a $d_h$-dimensional space, so very large $H$ makes each head
too narrow to represent anything — which is why $d_h$ sits at 64–128 in
essentially every production model, and $H$ grows with $d_{model}$.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp
from flax import nnx


class MultiHeadAttention(nnx.Module):
    def __init__(self, d_model: int, num_heads: int, *, rngs: nnx.Rngs):
        if d_model % num_heads != 0:
            raise ValueError(f"d_model={d_model} is not divisible by num_heads={num_heads}")

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads

        scale = 1.0 / jnp.sqrt(d_model)
        self.w_q = nnx.Param(jax.random.normal(rngs.params(), (d_model, d_model)) * scale)
        self.w_k = nnx.Param(jax.random.normal(rngs.params(), (d_model, d_model)) * scale)
        self.w_v = nnx.Param(jax.random.normal(rngs.params(), (d_model, d_model)) * scale)
        self.w_o = nnx.Param(jax.random.normal(rngs.params(), (d_model, d_model)) * scale)

    def _split_heads(self, x):
        """(B, T, d_model) -> (B, H, T, d_head)"""
        B, T, _ = x.shape
        # reshape splits the LAST axis into (H, d_head), then transpose moves
        # the head axis in front of the token axis.
        return x.reshape(B, T, self.num_heads, self.d_head).transpose(0, 2, 1, 3)

    def _merge_heads(self, x):
        """(B, H, T, d_head) -> (B, T, d_model)"""
        B, _, T, _ = x.shape
        return x.transpose(0, 2, 1, 3).reshape(B, T, self.d_model)

    def __call__(self, x, mask=None):
        q = self._split_heads(x @ self.w_q)
        k = self._split_heads(x @ self.w_k)
        v = self._split_heads(x @ self.w_v)

        # (B, H, T, d_head) @ (B, H, d_head, T) -> (B, H, T, T)
        scores = (q @ jnp.swapaxes(k, -1, -2)) / jnp.sqrt(
            jnp.asarray(self.d_head, q.dtype)
        )
        if mask is not None:
            scores = jnp.where(mask, scores, jnp.asarray(-1e9, scores.dtype))

        weights = jax.nn.softmax(scores, axis=-1)
        out = self._merge_heads(weights @ v)
        return out @ self.w_o

In [ ]:
# 🔍 Verify
import jax
import jax.numpy as jnp
from flax import nnx

mha = MultiHeadAttention(d_model=64, num_heads=8, rngs=nnx.Rngs(params=0))
x = jax.random.normal(jax.random.key(0), (2, 10, 64))
print("out:", mha(x).shape)

# Parameter count does not depend on the number of heads.
for h in (1, 2, 4, 8, 16):
    m = MultiHeadAttention(d_model=64, num_heads=h, rngs=nnx.Rngs(params=0))
    n = sum(int(jnp.size(leaf)) for leaf in jax.tree.leaves(nnx.state(m, nnx.Param)))
    print(f"H={h:2d}  d_head={64 // h:3d}  params={n}")

# A causal mask is just a boolean array broadcast over the head axis.
T = 10
causal = jnp.tril(jnp.ones((T, T), dtype=bool))
print("masked out:", mha(x, causal).shape)

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("mha")